# QRECAUDA en vivo: del circuito a la clave cifrada

Un recorrido de punta a punta, que corre en menos de dos minutos y sin red: circuito, transpilación a forma ISA, `SamplerV2`,
twirling de lectura, extracción (Peres + Toeplitz), batería NIST, AES-256-GCM y la tabla de métricas.

**Fuente de los bits.** Con `FUENTE = "aer"` todo el camino de IBM se ensaya contra el backend falso `fake_sherbrooke`: los bits los pone
Aer, que es pseudoaleatorio, y la salida lo rotula así. Con `FUENTE = "ibm"` los mismos pasos van al hardware real: exige un token **por ruta**
(`TOKEN_FILE`, nunca el valor), gasta cuota de QPU y puede esperar en cola. Lo medido a mano aquí NO es una medición preinscrita:
no decide ningún criterio (las mediciones de verdad están en `registro/`).

Este cuaderno es el «vivo»; `qrecauda.ipynb` es el anexo con las tablas del pitch.

In [1]:
# --- Parámetros (celda de parámetros) ---
FUENTE = "aer"        # "aer": ensayo contra un backend falso de IBM, sin credencial  |  "ibm": hardware real (gasta cuota)
TOKEN_FILE = ""       # sólo con FUENTE = "ibm": RUTA de un fichero con el token (nunca el token)
BACKEND = ""          # sólo con "ibm": nombre del backend; vacío = el menos ocupado con >= 8 qubits
MAX_SEGUNDOS_QPU = 60 # tope de QPU (estimado): aborta ANTES de enviar si lo pasa
SEMILLA = 20261007
QUBITS = 8
SHOTS = 16_000        # por trabajo (divisible por 8 PUBs)

In [2]:
import warnings
from dataclasses import replace

import numpy as np
from IPython.display import Markdown, display

assert FUENTE in ("aer", "ibm"), "FUENTE debe ser 'aer' o 'ibm'"
warnings.filterwarnings("ignore", category=DeprecationWarning)  # SamplerV2 está desaprobado en qiskit-ibm-runtime 0.50 (sigue funcionando)

from qrecauda import api
from qrecauda.aplicacion.demo import ROTULO_REAL, ROTULO_SIMULADO
from qrecauda.composicion import fuente_ibm_de
from qrecauda.transversal.configuracion import Configuracion

cfg = Configuracion(semilla=SEMILLA, ibm_modo="trabajo", ibm_backend=BACKEND, ibm_token_ruta=TOKEN_FILE)
print(f"fuente: {FUENTE}   semilla: {SEMILLA}   {QUBITS} qubits x {SHOTS} disparos")

fuente: aer   semilla: 20261007   8 qubits x 16000 disparos


## 1. El circuito

Ocho qubits en superposición (H) y medición de todos. El sesgo de lo que salga es el del ruido de lectura, no el del circuito.

In [3]:
from qiskit import QuantumCircuit

circuito = QuantumCircuit(QUBITS)
circuito.h(range(QUBITS))
circuito.measure_all()
print(circuito.draw("text"))

        ┌───┐ ░ ┌─┐                     
   q_0: ┤ H ├─░─┤M├─────────────────────
        ├───┤ ░ └╥┘┌─┐                  
   q_1: ┤ H ├─░──╫─┤M├──────────────────
        ├───┤ ░  ║ └╥┘┌─┐               
   q_2: ┤ H ├─░──╫──╫─┤M├───────────────
        ├───┤ ░  ║  ║ └╥┘┌─┐            
   q_3: ┤ H ├─░──╫──╫──╫─┤M├────────────
        ├───┤ ░  ║  ║  ║ └╥┘┌─┐         
   q_4: ┤ H ├─░──╫──╫──╫──╫─┤M├─────────
        ├───┤ ░  ║  ║  ║  ║ └╥┘┌─┐      
   q_5: ┤ H ├─░──╫──╫──╫──╫──╫─┤M├──────
        ├───┤ ░  ║  ║  ║  ║  ║ └╥┘┌─┐   
   q_6: ┤ H ├─░──╫──╫──╫──╫──╫──╫─┤M├───
        ├───┤ ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐
   q_7: ┤ H ├─░──╫──╫──╫──╫──╫──╫──╫─┤M├
        └───┘ ░  ║  ║  ║  ║  ║  ║  ║ └╥┘
meas: 8/═════════╩══╩══╩══╩══╩══╩══╩══╩═
                 0  1  2  3  4  5  6  7 


## 2. Transpilación a forma ISA

El hardware sólo entiende sus puertas nativas y sus qubits físicos. Se transpila contra el backend (aquí, el falso `fake_sherbrooke`; con `ibm`, el real).

In [4]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

from qrecauda.adaptadores.aer.transpilacion import a_isa

backend_modelo = FakeSherbrooke()
isa = a_isa(circuito, backend_modelo, semilla=SEMILLA)
print(f"backend: {backend_modelo.name}   vía: {isa.via} ({isa.motivo})")
print(f"profundidad: {isa.profundidad}   puertas de dos qubits: {isa.puertas_dos_qubits}   operaciones: {dict(isa.circuito.count_ops())}")

backend: fake_sherbrooke   vía: local (vía local: IA no pedida)
profundidad: 4   puertas de dos qubits: 0   operaciones: {'rz': 16, 'sx': 8, 'measure': 8, 'barrier': 1}


## 3. `SamplerV2` sobre Aer con el ruido del backend

La misma ISA, simulada con el ruido de lectura y de puertas que el backend declara en su calibración. Sesgo por qubit = p(1) − ½.

In [5]:
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as SamplerAer

sim = AerSimulator.from_backend(backend_modelo)
res = SamplerAer.from_backend(sim, default_shots=SHOTS, seed=SEMILLA).run([(isa.circuito, None, SHOTS)]).result()
nombre = isa.circuito.cregs[0].name
filas = np.array([[int(c) for c in s[::-1]] for s in res[0].data[nombre].get_bitstrings()], dtype=np.uint8)
sesgo_aer = filas.mean(axis=0) - 0.5
print("sesgo por qubit (Aer):", np.round(sesgo_aer, 4))
print(f"media |sesgo|: {np.abs(sesgo_aer).mean():.4f}")

sesgo por qubit (Aer): [ 0.0011 -0.0007  0.     -0.0013  0.0039 -0.0014  0.0083 -0.0049]
media |sesgo|: 0.0027


## 4. El camino de IBM: crudo frente a twirling

`FuenteIbm` envía **un trabajo** de 8 PUBs: uno crudo y siete con una X aleatoria por qubit antes de medir (que se deshace por XOR clásico).
El twirling de lectura cancela el sesgo de lectura asimétrico. Con `FUENTE = "aer"` esto es el ensayo (backend falso, sin cuota).

In [6]:
ibm = fuente_ibm_de(
    cfg, ensayo=(FUENTE == "aer"), mascaras=7, max_segundos_qpu=float(MAX_SEGUNDOS_QPU),
)
cruda, twirl = ibm.generar_contraste(QUBITS, SHOTS)
reg = ibm.ultimo_registro
sesgo = lambda m: m.bits.datos.reshape(m.qubits, m.shots).mean(axis=1) - 0.5
print(f"backend: {reg.backend}   job_id: {reg.job_id}   origen: {cruda.origen.value}   ensayo: {reg.ensayo}")
print(f"qubits físicos: {list(reg.qubits_fisicos)}   profundidad: {reg.profundidad}   calibración: {reg.calibracion_fecha}")
print("sesgo crudo     :", np.round(sesgo(cruda), 4), f" media |s| = {np.abs(sesgo(cruda)).mean():.4f}")
print("sesgo twirling  :", np.round(sesgo(twirl), 4), f" media |s| = {np.abs(sesgo(twirl)).mean():.4f}")

backend: fake_sherbrooke   job_id: e66f3eb5-5f68-4c68-9112-b94546f2dd07   origen: simulador_aer   ensayo: True
qubits físicos: [0, 1, 2, 3, 4, 5, 6, 7]   profundidad: 5   calibración: 2025-02-26T14:43:10-05:00
sesgo crudo     : [-0.016   0.018   0.001   0.021   0.0085  0.007   0.2705  0.026 ]  media |s| = 0.0460
sesgo twirling  : [ 0.0005 -0.0025  0.0057  0.0021 -0.0082  0.0016 -0.0419  0.0014]  media |s| = 0.0080


## 5. Extracción, NIST y clave

Peres (von Neumann recursivo) y Toeplitz dimensionados por la min-entropía de la muestra; después la batería NIST (o el validador estadístico si el binario no está) sobre la clave.

In [7]:
from qrecauda.aplicacion.pipeline import ParametrosPipeline, ejecutar
from qrecauda.composicion import validador_de, servicio_de
from qrecauda.dominio.errores import FuenteNoDisponible
from qrecauda.transversal.observabilidad import RelojMonotonico

try:
    validador = validador_de(Configuracion(validador="nist"))
except FuenteNoDisponible as e:
    print("NIST no disponible, se usa el validador estadístico:", e)
    validador = validador_de(Configuracion())


class Reproduce:
    """Entrega la muestra ya obtenida al pipeline (es la fuente: no genera nada)."""

    def __init__(self, m):
        self.m = m

    def generar(self, qubits, shots):
        return self.m


r = ejecutar(Reproduce(twirl), validador, RelojMonotonico(), ParametrosPipeline(QUBITS, twirl.shots))
print(f"bits crudos: {r.bits_crudos}  ->  tras Peres: {r.bits_extraidos}  ->  clave: {len(r.clave)} bits   min-entropía de entrada: {r.h_min:.4f}")
for etapa, medidas in r.etapas:
    print(f"  {etapa:<8}", {m.metrica.name: round(m.valor, 4) for m in medidas})
print("clave aprobada (M1-M5):", r.veredicto.calidad_de_clave_aprobada)

bits crudos: 112000  ->  tras Peres: 98216  ->  clave: 32331 bits   min-entropía de entrada: 0.9873
  cruda    {'SESGO': 0.0052, 'MONOBIT': 0.0006, 'RUNS': 0.0, 'CHI2': 0.0}
  clave    {'SESGO': 0.0009, 'MONOBIT': 0.7512, 'RUNS': 0.8548, 'CHI2': 0.148}
clave aprobada (M1-M5): True


## 6. AES-256-GCM

Un peaje y un trayecto de Metro, cifrados y descifrados con la clave de arriba (si aprobó M1-M5; si no, no se cifra nada).

In [8]:
from qrecauda.aplicacion.demo import METRO, PEAJE

if r.veredicto.calidad_de_clave_aprobada:
    servicio = servicio_de(cfg, r)
    for nombre, tx in (("peaje", PEAJE), ("metro", METRO)):
        sellada = servicio.cifrar(tx)
        print(f"{nombre}: {len(sellada.cifrado)} B cifrados  nonce {sellada.nonce.a_bytes().hex()}  descifrado OK: {servicio.descifrar(sellada) == tx}")
        print("  rótulo:", sellada.rotulo)
else:
    print("la clave no aprobó M1-M5: no se cifra nada")

peaje: 102 B cifrados  nonce 494ad9206ff6798e0c759dc1  descifrado OK: True
  rótulo: validación del pipeline
metro: 101 B cifrados  nonce 216d1a1a77ddded0fb9ff71c  descifrado OK: True
  rótulo: validación del pipeline


## 7. Tabla de métricas de la demo

Las tres ramas lado a lado (PRNG, Aer sin mitigar, Aer con twirling) con M1, min-entropía, tres p-valores NIST y bits de clave. `qrecauda demo --fuente ibm` añade las filas de hardware; aquí el hardware es el de los pasos 4 y 5.

In [9]:
from qrecauda.presentacion import tabla_de_demo

d = api.demo(cfg, rapido=True)  # las tres ramas locales; las de hardware son las del paso 4-5 (no se vuelve a gastar cuota)
print(tabla_de_demo(d))

QRECAUDA demo  (8 qubits, 40000 disparos, fuente aer, rápido)

fuente                            M1 cruda  M1 mitig  M1 clave  min-ent  p mono  p runs  p chi2   bits  M1-M5
PRNG clasico                        0.0004         -    0.0011   0.9847   0.480   0.529   0.986  95228  APRUEBA
Aer sin mitigar (ruido medio)       0.0298         -    0.0018   0.9829   0.274   0.990   0.971  95085  APRUEBA
Aer con twirling (ruido medio)      0.0298    0.0009    0.0020   0.9822   0.214   0.862   0.065  94875  APRUEBA

Origen de cada fuente:
  PRNG clasico: clásico: PRNG determinista dada la semilla
  Aer sin mitigar (ruido medio): simulado: Aer es pseudoaleatorio, sin origen cuántico
  Aer con twirling (ruido medio): simulado: Aer es pseudoaleatorio, sin origen cuántico

AES-256-GCM con la clave de «Aer con twirling (ruido medio)»:
  peaje  102 B cifrados  nonce 80c2c4797b7d3817f584e879  8a5bbbc190475d693ac76a745b002945…  descifrado OK  [validación del pipeline]
  metro  101 B cifrados  nonce 8697c1

## Qué se puede y qué no se puede decir

- La batería NIST y la cota de min-entropía miden **estadística, no origen**: un PRNG las pasa igual (control N1).
- Si el rótulo dice «simulado», los bits los puso Aer; sólo un `job_id` real de IBM cambia eso.
- Para el hardware real: `docs/HARDWARE.md` y `scripts/ibm_run.sh`.